# 01 · Hello MCP : mon premier outil appelé par une IA

**Projet Synapse** · Phase 1 · Semaine 1

## Objectif
Créer un tout petit serveur MCP avec un outil, puis l'appeler depuis ce notebook, **sans LLM**. À la fin, tu auras vu de tes yeux comment un outil est exposé et appelé.

## Ce qu'on va faire
1. Vérifier l'environnement
2. Écrire un serveur MCP (fichier `servers/hello_server.py`)
3. Se connecter en client et lister les outils
4. Appeler un outil
5. Regarder ce que le LLM « voit » d'un outil
6. Exercice : ajouter ton propre outil

## 0 · L'idée en 30 secondes

| Rôle | C'est quoi | Ici |
|---|---|---|
| **Serveur MCP** | Un programme qui expose des outils | `hello_server.py` |
| **Client MCP** | Le code qui se connecte au serveur et appelle ses outils | ce notebook (plus tard : l'agent) |
| **Outil (tool)** | Une fonction avec un nom, une description et des paramètres | `donner_heure` |

Le serveur tourne dans **son propre processus**. Le client le lance et lui parle via ses entrées/sorties (`stdio`). Le LLM, plus tard, ne fera que *choisir* quel outil appeler : c'est le client qui exécute.

## 1 · Vérifier l'environnement

In [1]:
import sys, os
print("Python :", sys.version.split()[0])
print("Exécutable :", sys.executable)
print("Dossier courant :", os.getcwd())

Python : 3.13.9
Exécutable : c:\Users\nasri\OneDrive\Desktop\project_5\multioutilsMCP_AI\env_synapse\Scripts\python.exe
Dossier courant : c:\Users\nasri\OneDrive\Desktop\project_5\multioutilsMCP_AI\notebooks


>  L'exécutable doit contenir `env_synapse`. Sinon, choisis le kernel **Python (synapse)** en haut à droite.
>  Le dossier courant doit se terminer par `notebooks`.

In [1]:
%pip install -q tzdata
from importlib.metadata import version
print("mcp", version("mcp"))

Note: you may need to restart the kernel to use updated packages.
mcp 2.3.0


`tzdata` est nécessaire sous **Windows** pour gérer les fuseaux horaires (Windows n'a pas la base de données de fuseaux de Linux/Mac).

 **Version du SDK** : on utilise `mcp` en version 2 (`MCPServer`). Beaucoup de tutoriels en ligne montrent l'ancienne version 1 (`FastMCP`), dont le code ne marche pas tel quel ici. Si tu vois une erreur d'import, c'est probablement ça.

## 2 · Écrire le serveur

Un outil = une fonction Python avec :
- des **types** sur les paramètres (le SDK en fait un schéma automatiquement),
- une **docstring** : c'est la description que le LLM lira pour décider *quand* utiliser l'outil. Elle est donc très importante.

La cellule ci-dessous écrit le fichier `../servers/hello_server.py`.

In [2]:
%%writefile ../servers/hello_server.py
from datetime import datetime
from zoneinfo import ZoneInfo

from mcp.server.mcpserver import MCPServer

mcp = MCPServer("hello-synapse")


@mcp.tool()
def donner_heure(fuseau: str = "Africa/Casablanca") -> str:
    """Donne la date et l'heure actuelles dans un fuseau horaire.
    Exemples de fuseaux : Africa/Casablanca, Europe/Paris, America/New_York."""
    return datetime.now(ZoneInfo(fuseau)).strftime("%A %d %B %Y, %H:%M:%S")


@mcp.tool()
def dire_bonjour(nom: str, langue: str = "fr") -> str:
    """Salue une personne par son prénom, en français (fr) ou en anglais (en)."""
    return f"Hello {nom} !" if langue == "en" else f"Bonjour {nom} !"


if __name__ == "__main__":
    mcp.run()  # démarre le serveur en mode stdio

Overwriting ../servers/hello_server.py


> On ne lance **pas** ce fichier à la main : c'est le client qui va le démarrer.
> Si tu le lances dans un terminal, il semble « bloqué » : c'est normal, il attend qu'un client lui parle (`Ctrl + C` pour l'arrêter).

## 3 · Se connecter en client et lister les outils

On décrit comment lancer le serveur (`python servers/hello_server.py`), puis on se connecte. Dans un notebook on peut écrire `await` directement.

On redirige les messages d'erreur du serveur vers un fichier log pour que le notebook reste propre (et pour éviter un souci connu avec `stderr` dans Jupyter).

In [4]:
import sys
from pathlib import Path
from mcp.client import Client
from mcp.client.stdio import StdioServerParameters, stdio_client

SERVER_PATH = Path("../servers/hello_server.py").resolve()
LOG_PATH = Path("../servers/hello_server.log")

def ouvrir_client():
    params = StdioServerParameters(command=sys.executable, args=[str(SERVER_PATH)])
    return Client(stdio_client(params, errlog=open(LOG_PATH, "w")))

async with ouvrir_client() as client:
    resultat = await client.list_tools()

for outil in resultat.tools:
    print(f"• {outil.name}\n    {outil.description}\n")

• donner_heure
    Donne la date et l'heure actuelles dans un fuseau horaire.
Exemples de fuseaux : Africa/Casablanca, Europe/Paris, America/New_York.

• dire_bonjour
    Salue une personne par son prénom, en français (fr) ou en anglais (en).



## 4 · Appeler un outil

Le client envoie le **nom** de l'outil et ses **arguments** ; le serveur exécute et renvoie le résultat.
Plus tard, c'est le LLM qui décidera du nom et des arguments.

In [5]:
async def appeler(nom_outil, arguments=None):
    async with ouvrir_client() as client:
        reponse = await client.call_tool(nom_outil, arguments or {})
    return reponse

r1 = await appeler("donner_heure")
print("Casablanca :", r1.content[0].text)

r2 = await appeler("donner_heure", {"fuseau": "America/New_York"})
print("New York   :", r2.content[0].text)

r3 = await appeler("dire_bonjour", {"nom": "Sara", "langue": "en"})
print(r3.content[0].text)

Casablanca : Sunday 18 October 2026, 23:10:39
New York   : Sunday 18 October 2026, 19:10:43
Hello Sara !


**Ce qu'on a vu :** on a appelé des fonctions Python *à travers un protocole*, sans les importer. Le serveur pourrait être écrit dans un autre langage ou tourner sur une autre machine : le client s'en moque.

## 5 · Ce que le LLM voit vraiment

Le LLM ne voit pas ton code. Il voit seulement, pour chaque outil : le **nom**, la **description** et le **schéma des paramètres**.
Une mauvaise description = un mauvais choix d'outil. C'est l'un des risques de notre cahier des charges.

In [6]:
import json

async with ouvrir_client() as client:
    resultat = await client.list_tools()

for outil in resultat.tools:
    vue_llm = {
        "name": outil.name,
        "description": outil.description,
        "input_schema": outil.input_schema,
    }
    print(json.dumps(vue_llm, indent=2, ensure_ascii=False))
    print("-" * 60)

{
  "name": "donner_heure",
  "description": "Donne la date et l'heure actuelles dans un fuseau horaire.\nExemples de fuseaux : Africa/Casablanca, Europe/Paris, America/New_York.",
  "input_schema": {
    "type": "object",
    "properties": {
      "fuseau": {
        "default": "Africa/Casablanca",
        "title": "Fuseau",
        "type": "string"
      }
    },
    "title": "donner_heureArguments"
  }
}
------------------------------------------------------------
{
  "name": "dire_bonjour",
  "description": "Salue une personne par son prénom, en français (fr) ou en anglais (en).",
  "input_schema": {
    "type": "object",
    "properties": {
      "nom": {
        "title": "Nom",
        "type": "string"
      },
      "langue": {
        "default": "fr",
        "title": "Langue",
        "type": "string"
      }
    },
    "required": [
      "nom"
    ],
    "title": "dire_bonjourArguments"
  }
}
------------------------------------------------------------


## 6 · Que se passe-t-il quand ça se passe mal ?

Un agent doit gérer les erreurs. Regardons ce qui revient avec un fuseau qui n'existe pas.

In [7]:
try:
    r = await appeler("donner_heure", {"fuseau": "Mars/Olympus"})
    print("is_error :", r.is_error)
    print("Message  :", r.content[0].text)
except Exception as e:
    print("Exception :", type(e).__name__, e)

is_error : True
Message  : Error executing tool donner_heure


 **À noter pour la suite** : une erreur d'outil doit être renvoyée proprement à l'agent (pour qu'il corrige ou abandonne), jamais faire planter toute l'application.

## 7 · Exercice : ton propre outil

1. Ouvre `servers/hello_server.py`.
2. Ajoute un outil, par exemple :
   - `convertir_devise(montant: float, de: str, vers: str)` avec des taux fixes écrits dans le code,
   - `jours_avant(date_iso: str)` : nombre de jours avant une date,
   - `compter_mots(texte: str)`.
3. Soigne la **docstring** : elle doit dire clairement *quand* utiliser l'outil.
4. Relance la cellule 3 : ton outil doit apparaître dans la liste.
5. Appelle-le dans une nouvelle cellule avec `await appeler("ton_outil", {...})`.

**Questions à discuter à deux :**
- Que se passe-t-il si la docstring est vague ?
- Quel outil serait dangereux à donner à une IA (supprimer, envoyer, payer…) ? Comment le protéger ? (→ niveaux de risque, Phase 4)

## 8 · Bonus : tester avec MCP Inspector (optionnel)

MCP Inspector est l'outil officiel avec interface web pour tester un serveur. Il demande **Node.js** installé. Dans un terminal, depuis la racine du projet, avec `env_synapse` activé :

```powershell
npx @modelcontextprotocol/inspector python servers/hello_server.py
```

Une page s'ouvre dans le navigateur : tu peux lister les outils et les appeler à la main.

## Récapitulatif
- Un **serveur MCP** expose des outils (nom + description + paramètres).
- Un **client** les liste et les appelle.
- Le **LLM** ne verra que la description : elle compte autant que le code.
- Prochain notebook : **02 · brancher un LLM** pour qu'il choisisse et appelle l'outil tout seul.